In [6]:
# 全列の、Excelデータの型と欠損値があるかどうかを調べる
import pandas as pd
import numpy as np

# ①国交省HPからDLした建設関連業の登録業者(20260411にDL実施)のExcelファイルを読み込み
df = pd.read_excel('建設関連業の登録業者(測量業者_260331).xlsx')

# ②列名ごとに、型と欠損値があるかどうか調査
col_chk_data_df =pd.DataFrame()
for col in df.columns:
    empty_row = pd.DataFrame(['列名' , '型','欠損値の有無'])
    empty_row_col = pd.DataFrame([{}])
    col_col = df[col]
    col_chk_data_df = pd.concat([col_col, empty_row_col], ignore_index=False)
    empty_row_type = pd.DataFrame([{}])
    col_type = df[col].dtype
    col_chk_data_df = pd.concat([col_type,  empty_row_type], ignore_index=False)
    empty_row_null = pd.DataFrame([{}])
    col_null = df[col].isnull().any()
    col_chk_data_df = pd.concat([col_null, empty_row_null], ignore_index=False)

# ③Excelファイルへ調査結果を書き込む
writer = pd.ExcelWriter('建設関連業の登録業者(測量業者_260331)_型chk_all2.xlsx')
col_chk_data_df.to_excel(writer,sheet_name='型とNullの検査', index=False)
writer.close()

TypeError: cannot concatenate object of type '<class 'numpy.dtypes.Int64DType'>'; only Series and DataFrame objs are valid

In [4]:
# 香川県内に本社・本店・支社・営業所等がある会社(協同組合や社団法人は含めない)を抽出
# 本社・本店の行の値を、役員兼＿技術＿計、その他＿技術＿計、法人＿完成業務収入の空白行に反映させる
import pandas as pd
import numpy as np

# ①国交省HPからDLした建設関連業の登録業者(20260411にDL実施)のExcelファイルを読み込み
df = pd.read_excel('建設関連業の登録業者(測量業者_260331).xlsx')

# ②前準備
def exclude_kagawa_business_store_from_df(df):

    # 必要な列だけコピー
    copy_df = df.loc[:, ['通し番号','行番号','出力日','登録番号','商号又は名称かな',
        '商号又は名称','資本金','会社種別コード','測量の種類＿三角','測量の種類＿多角',
        '測量の種類＿水準','測量の種類＿地形','測量の種類＿空撮','測量の種類＿空図',
        '測量の種類＿地図','測量の種類＿その他','営業所区分コード','営業所区分名',
        '営業所名','営業所所在地','役員兼＿技術＿計','その他＿技術＿計',
        '法人＿完成業務収入']]

    # 営業所区分コードに欠損値を補い、数値型に変換
    copy_df['営業所区分コード'] = pd.to_numeric(copy_df['営業所区分コード'], errors='coerce')

    # 会社種別コードが2,3,4,5のもの(協同組合や社団法人等の「会社」ではないものを除外)を抽出
    company_code_df = copy_df[copy_df['会社種別コード'].isin([2,3,4,5])]

    # ③役員兼＿技術＿計、その他＿技術＿計、法人＿完成業務収入の空白行を埋める
     # 営業所所在地の項目に香川県の記載があるものを抽出(営業所区分コードが1以外も含む)
     # ⇒香川県に住所があるもののみ抽出
     # (kagawa_main_branch_store_df：以下、香川県のみに住所dfと表記)
    kagawa_main_branch_store_df \
    = company_code_df[company_code_df['営業所所在地'] .str.contains('香川県',na=False)]


     # 本社や本店だけ(営業所区分コードが1)で、営業所所在地が香川県以外
     # ⇒本社や本店が香川県外にあるもののみ抽出
     # (main_store_no_kagawa_df：以下、香川県以外に本社・本店dfと表記)
    main_store_no_kagawa_df \
    = company_code_df[(company_code_df['営業所区分コード']==1) \
      &(~company_code_df['営業所所在地'].str.contains('香川県',na=False))]

      # 香川県のみに住所dfと香川県以外に本社・本店dfを比較
    for index1, row1 in kagawa_main_branch_store_df \
     [['登録番号','役員兼＿技術＿計','その他＿技術＿計','法人＿完成業務収入']].iterrows():
        for index2, row2 in main_store_no_kagawa_df \
         [['登録番号','役員兼＿技術＿計','その他＿技術＿計','法人＿完成業務収入']].iterrows():

        # 香川県のみに住所dfの法人＿完成業務収入の列でNaNのものを特定
            if pd.isna(kagawa_main_branch_store_df(['法人＿完成業務収入'])):

            # 香川県のみに住所dfの登録番号を、香川県以外に本社・本店dfの登録番号と比較
                if kagawa_main_branch_store_df[index1,'登録番号'] \
                   == main_store_no_kagawa_df[index2,'登録番号']:

                    # 登録番号が同じであれば、香川県以外に本社・本店dfの役員兼＿技術＿計、
                    # その他＿技術＿計、法人＿完成業務収入の数値を香川県のみに住所dfに反映
                    kagawa_main_branch_store_df.at[index1,'役員兼＿技術＿計'] \
                    =  main_store_no_kagawa_df[index2,'役員兼＿技術＿計']
                    kagawa_main_branch_store_df.at[index1,'その他＿技術＿計'] \
                    =  main_store_no_kagawa_df[index2,'その他＿技術＿計']
                    kagawa_main_branch_store_df.at[index1,'法人＿完成業務収入'] \
                    =  main_store_no_kagawa_df[index2,'法人＿完成業務収入']


    # 役員兼＿技術＿計、その他＿技術＿計、法人＿完成業務収入を反映させ香川県のみに住所dfを
    # kagawa_main_branch_store_complete_df(以下、香川県のみに住所df_完成版と表記)に代入
    kagawa_main_branch_store_complete_df = kagawa_main_branch_store_df

    # ④3つの結果をExcelファイルに書き込む
    with pd.ExcelWriter('建設関連業の登録業者(測量業者_260331)_香川県のみ営業所_値反映.xlsx') as writer:
        kagawa_main_branch_store_df.to_excel(writer, sheet_name='香川県のみに住所df', index=False)
        main_store_no_kagawa_df.to_excel(writer, sheet_name='香川県以外に本社・本店df', index=False)
        kagawa_main_branch_store_complete_df.to_excel(writer, sheet_name='香川県のみに住所df_完成版', index=False)
        return kagawa_main_branch_store_df, main_store_no_kagawa_df, kagawa_main_branch_store_complete_df

# ⑤関数を呼び出して実行する
exclude_kagawa_business_store_from_df(df)

TypeError: 'DataFrame' object is not callable

In [ ]:
# 香川県内の測量会社の完成業務収入(建設会社における売上高に該当する項目)を集計
import pandas as pd

# ①前準備
#整理したExcelファイルを読み込んで、データフレーム化
df = pd.read_excel('建設関連業の登録業者(測量業者_260331)_香川県のみ営業所_値反映.xlsx')

# 必要な列のみをコピー
new_sheet1 = df.loc[:, ['商号又は名称', '商号又は名称かな',
    '法人＿完成業務収入', '役員兼＿技術＿計','その他＿技術＿計']]

# 列名を変更
new_sheet1 = new_sheet1.rename(columns={'法人＿完成業務収入':'完成業務収入(円)',
    '役員兼＿技術＿計':'技術社員(役員)(人)','その他＿技術＿計':'技術社員(役員以外)(人)'})

# 技術職員(人)=技術職員(役員)＋技術職員(役員以外)
new_sheet1['全技術社員(人)'] = new_sheet1['技術社員(役員)(人)']+new_sheet1['技術社員(役員以外)(人)']

# 空のシートを作成
# シート2は、香川県の会社全体の完成業務収入に対する各会社のシェア率を算出
new_sheet2 = pd.DataFrame()
new_sheet2 = new_sheet1.loc[:, ['商号又は名称', '商号又は名称かな','完成業務収入(円)']]
# シート3は、技術社員1人あたりの各会社の完成業務収入を算出
new_sheet3 = pd.DataFrame()
new_sheet3 = new_sheet1.loc[:, ['商号又は名称', '商号又は名称かな','完成業務収入(円)','全技術社員(人)']]

# ②集計
# 完成業務収入(completed_contract_revenue)の合計
completed_contract_revenue = new_sheet2['完成業務収入(円)'].sum()

# ③シェア算出・順位付け
# 会社ごとの完成業務収入のシェア
new_sheet2['シェア率(%)'] = (new_sheet2['完成業務収入(円)']/completed_contract_revenue) * 100
# 完成業務収入のシェア率による順位付け
new_sheet2['順位'] = new_sheet2['シェア率(%)'].rank(ascending=False)
new_sheet2['順位'] = new_sheet2['順位'].astype(int)

# シート2の一番下の行に、完成業務収入の合計値を書き込む
# 空白行を1行分作成する
empty_row2 = pd.DataFrame([{}])
new_sheet2 = pd.concat([new_sheet2, empty_row2], ignore_index=False)
# 合計値を書き込む
new_sheet2.at[len(new_sheet2)+1, '商号又は名称かな'] = '完成業務収入合計(円)'
new_sheet2.at[len(new_sheet2), '完成業務収入(円)'] = completed_contract_revenue

# 技術社員1人ごとの完成業務収入ランキング
# 技術社員1人ごとの完成業務収入算出
new_sheet3['1人あたりの完成業務収入(円)'] = (new_sheet3['完成業務収入(円)']/new_sheet3['全技術社員(人)'])
# 技術社員1人あたりの完成業務収入による順位付け
new_sheet3['順位'] = new_sheet3['1人あたりの完成業務収入(円)'] .rank(ascending=False)
new_sheet3['順位'] = new_sheet3['順位'].astype(int)

# シート3の一番下の行に、完成業務収入の合計値を書き込む
# 空白行を1行分作成する
empty_row3 = pd.DataFrame([{}])
# 合計値を書き込む
new_sheet3 = pd.concat([new_sheet3, empty_row3], ignore_index=False)
new_sheet3.at[len(new_sheet3)+1, '商号又は名称かな'] = '完成業務収入合計(円)'
new_sheet3.at[len(new_sheet3), '完成業務収入(円)'] = completed_contract_revenue

# ④新規Excelファイル作成・書込
with pd.ExcelWriter('建設関連業の登録業者(測量業者_260331)_香川県のみ営業所_値反映_シェア_p.xlsx') as writer:
    new_sheet1.to_excel(writer,sheet_name='香川県内の会社一覧_整理', index=False)
    new_sheet2.to_excel(writer,sheet_name='完成業務収入のシェア率', index=False)
    new_sheet3.to_excel(writer,sheet_name='技術社員1人当たりの完成業務収入', index=False)